In [1]:
import json
from typing import List, Dict, Optional, Tuple, Set
from dataclasses import dataclass, field
from collections import defaultdict
from enum import Enum


class MasteryLevel(Enum):
    """Mastery level for a belt"""
    NOT_ASSESSED = "not_assessed"
    NEEDS_FULL_COURSE = "needs_full_course"      # < 40%
    NEEDS_REVIEW = "needs_review"                 # 40-69%
    MASTERED = "mastered"                         # 70-89%
    FULLY_MASTERED = "fully_mastered"            # >= 90%


@dataclass
class ConceptScore:
    """Score for a specific concept"""
    concept: str
    correct: int
    total: int
    percentage: float
    mastered: bool


@dataclass
class BeltAssessment:
    """Assessment result for a single belt"""
    belt: str
    total_questions: int
    correct_answers: int
    percentage: float
    weighted_score: float
    max_weighted_score: float
    mastery_level: MasteryLevel

    # Breakdown
    by_difficulty: Dict[int, Dict[str, int]] = field(default_factory=dict)
    by_concept: Dict[str, ConceptScore] = field(default_factory=dict)

    # Insights
    strong_concepts: List[str] = field(default_factory=list)
    weak_concepts: List[str] = field(default_factory=list)

    @property
    def needs_study(self) -> bool:
        return self.mastery_level in [
            MasteryLevel.NEEDS_FULL_COURSE,
            MasteryLevel.NEEDS_REVIEW
        ]

    @property
    def can_skip(self) -> bool:
        return self.mastery_level in [
            MasteryLevel.MASTERED,
            MasteryLevel.FULLY_MASTERED
        ]


@dataclass
class PlacementDecision:
    """Final placement decision for the student"""

    # Core decision
    belts_to_study: List[str]           # Belts student needs to take
    belts_to_skip: List[str]            # Belts student can skip
    belts_to_review: List[str]          # Belts needing light review only

    # Priority order
    study_priority: List[Dict]          # Ordered list with reasons

    # Overall stats
    total_belts_assessed: int
    overall_readiness: float            # Overall course readiness %

    # Detailed assessments
    belt_assessments: Dict[str, BeltAssessment]

    # Personalized insights
    overall_strengths: List[str]
    overall_weaknesses: List[str]
    study_plan: List[str]

    # Raw data for further processing
    detailed_analysis: Dict


class IndependentBeltEvaluator:
    """
    Evaluates placement test where each belt is an independent topic/skill area.

    Key Differences from Sequential Model:
    - Each belt assessed independently
    - Student may need some belts but not others
    - No assumption that mastering Belt A means ready for Belt B
    - Recommendations focus on WHICH belts to study, not WHICH level
    """

    DIFFICULTY_WEIGHTS = {1: 1, 2: 2, 3: 3}
    DIFFICULTY_LABELS = {1: 'Easy', 2: 'Medium', 3: 'Hard'}

    # Thresholds for mastery levels
    THRESHOLDS = {
        'fully_mastered': 0.90,    # >= 90%
        'mastered': 0.70,          # >= 70%
        'needs_review': 0.40,      # >= 40%
        # Below 40% = needs full course
    }

    def __init__(
        self,
        mastery_threshold: float = 0.70,
        skip_threshold: float = 0.70,
        review_threshold: float = 0.40,
        use_weighted_scoring: bool = True,
        difficulty_weights: Optional[Dict[int, int]] = None,
        concept_mastery_threshold: float = 0.70
    ):
        """
        Initialize the evaluator.

        Args:
            mastery_threshold: Minimum % to consider a belt mastered
            skip_threshold: Minimum % to skip a belt entirely
            review_threshold: Below this = needs full course, above = just review
            use_weighted_scoring: Weight harder questions more
            difficulty_weights: Custom weights {1: easy, 2: medium, 3: hard}
            concept_mastery_threshold: Minimum % to master a concept
        """
        self.mastery_threshold = mastery_threshold
        self.skip_threshold = skip_threshold
        self.review_threshold = review_threshold
        self.use_weighted_scoring = use_weighted_scoring
        self.difficulty_weights = difficulty_weights or self.DIFFICULTY_WEIGHTS
        self.concept_mastery_threshold = concept_mastery_threshold

    def _determine_mastery_level(self, percentage: float) -> MasteryLevel:
        """Determine mastery level based on percentage score"""
        if percentage >= self.THRESHOLDS['fully_mastered']:
            return MasteryLevel.FULLY_MASTERED
        elif percentage >= self.THRESHOLDS['mastered']:
            return MasteryLevel.MASTERED
        elif percentage >= self.THRESHOLDS['needs_review']:
            return MasteryLevel.NEEDS_REVIEW
        else:
            return MasteryLevel.NEEDS_FULL_COURSE

    def _assess_belt(
        self,
        questions: List[Dict],
        answers: List[int],
        belt: str
    ) -> BeltAssessment:
        """Assess a single belt independently"""

        # Filter questions for this belt
        belt_data = [
            (q, a) for q, a in zip(questions, answers)
            if q.get('belt') == belt
        ]

        if not belt_data:
            return BeltAssessment(
                belt=belt,
                total_questions=0,
                correct_answers=0,
                percentage=0.0,
                weighted_score=0.0,
                max_weighted_score=0.0,
                mastery_level=MasteryLevel.NOT_ASSESSED
            )

        # Calculate scores
        total = len(belt_data)
        correct = 0
        weighted_score = 0.0
        max_weighted_score = 0.0

        by_difficulty = defaultdict(lambda: {'correct': 0, 'total': 0})
        concept_stats = defaultdict(lambda: {'correct': 0, 'total': 0})

        for question, answer in belt_data:
            difficulty = question.get('difficulty_level', 1)
            concepts = question.get('concepts', [])
            correct_idx = question.get('ans_idx')
            is_correct = (answer == correct_idx)

            weight = self.difficulty_weights.get(difficulty, 1)
            max_weighted_score += weight

            if is_correct:
                correct += 1
                weighted_score += weight

            # Track by difficulty
            by_difficulty[difficulty]['total'] += 1
            if is_correct:
                by_difficulty[difficulty]['correct'] += 1

            # Track by concept
            for concept in concepts:
                concept_stats[concept]['total'] += 1
                if is_correct:
                    concept_stats[concept]['correct'] += 1

        # Calculate percentage
        if self.use_weighted_scoring and max_weighted_score > 0:
            percentage = weighted_score / max_weighted_score
        else:
            percentage = correct / total if total > 0 else 0.0

        # Assess concepts
        by_concept = {}
        strong_concepts = []
        weak_concepts = []

        for concept, stats in concept_stats.items():
            if stats['total'] > 0:
                concept_pct = stats['correct'] / stats['total']
                concept_mastered = concept_pct >= self.concept_mastery_threshold

                by_concept[concept] = ConceptScore(
                    concept=concept,
                    correct=stats['correct'],
                    total=stats['total'],
                    percentage=concept_pct,
                    mastered=concept_mastered
                )

                if concept_pct >= 0.80:
                    strong_concepts.append(concept)
                elif concept_pct < 0.50:
                    weak_concepts.append(concept)

        # Determine mastery level
        mastery_level = self._determine_mastery_level(percentage)

        return BeltAssessment(
            belt=belt,
            total_questions=total,
            correct_answers=correct,
            percentage=percentage,
            weighted_score=weighted_score,
            max_weighted_score=max_weighted_score,
            mastery_level=mastery_level,
            by_difficulty=dict(by_difficulty),
            by_concept=by_concept,
            strong_concepts=strong_concepts,
            weak_concepts=weak_concepts
        )

    def _calculate_study_priority(
        self,
        assessments: Dict[str, BeltAssessment]
    ) -> List[Dict]:
        """
        Calculate priority order for studying belts.

        Priority factors:
        1. Lower score = higher priority (needs more work)
        2. More weak concepts = higher priority
        3. Failed hard questions = might need fundamentals
        """
        priority_list = []

        for belt, assessment in assessments.items():
            if not assessment.needs_study:
                continue

            # Calculate priority score (higher = more urgent)
            priority_score = 0
            reasons = []

            # Factor 1: Inverse of percentage (lower score = higher priority)
            priority_score += (1 - assessment.percentage) * 50

            # Factor 2: Number of weak concepts
            weak_count = len(assessment.weak_concepts)
            if weak_count > 0:
                priority_score += weak_count * 10
                reasons.append(f"{weak_count} concepts need work")

            # Factor 3: Performance on easy questions
            easy_stats = assessment.by_difficulty.get(
                1, {'correct': 0, 'total': 0})
            if easy_stats['total'] > 0:
                easy_pct = easy_stats['correct'] / easy_stats['total']
                if easy_pct < 0.70:
                    priority_score += 20
                    reasons.append("Fundamentals need attention")

            # Factor 4: Mastery level
            if assessment.mastery_level == MasteryLevel.NEEDS_FULL_COURSE:
                priority_score += 15
                reasons.append("Needs comprehensive study")
            else:
                reasons.append("Needs review/reinforcement")

            priority_list.append({
                'belt': belt,
                'priority_score': priority_score,
                'percentage': assessment.percentage,
                'mastery_level': assessment.mastery_level.value,
                'reasons': reasons,
                'weak_concepts': assessment.weak_concepts
            })

        # Sort by priority score (descending)
        priority_list.sort(key=lambda x: x['priority_score'], reverse=True)

        # Add rank
        for i, item in enumerate(priority_list, 1):
            item['rank'] = i

        return priority_list

    def _generate_study_plan(
        self,
        assessments: Dict[str, BeltAssessment],
        study_priority: List[Dict]
    ) -> List[str]:
        """Generate personalized study plan recommendations"""

        plan = []

        if not study_priority:
            plan.append(
                "🎉 Excellent! You've demonstrated mastery across all assessed belts.")
            plan.append(
                "Consider exploring advanced topics or helping other learners.")
            return plan

        # Opening recommendation
        needs_full = [
            p['belt'] for p in study_priority
            if p['mastery_level'] == 'needs_full_course'
        ]
        needs_review = [
            p['belt'] for p in study_priority
            if p['mastery_level'] == 'needs_review'
        ]

        if needs_full:
            plan.append(
                f"📚 Start with complete courses for: {', '.join(needs_full)}"
            )

        if needs_review:
            plan.append(
                f"📖 Review materials recommended for: {', '.join(needs_review)}"
            )

        # Concept-specific recommendations
        all_weak_concepts = set()
        for p in study_priority:
            all_weak_concepts.update(p.get('weak_concepts', []))

        if all_weak_concepts:
            plan.append(
                f"🎯 Focus especially on these concepts: {', '.join(list(all_weak_concepts)[:5])}"
            )

        # Priority recommendation
        if study_priority:
            top_priority = study_priority[0]
            plan.append(
                f"⭐ Recommended to start with: {top_priority['belt']} "
                f"(Score: {top_priority['percentage']:.0%})"
            )

        return plan

    def _collect_overall_insights(
        self,
        assessments: Dict[str, BeltAssessment]
    ) -> Tuple[List[str], List[str]]:
        """Collect overall strengths and weaknesses across all belts"""

        strengths = []
        weaknesses = []

        # Collect from all assessments
        all_strong_concepts = []
        all_weak_concepts = []

        mastered_belts = []
        struggling_belts = []

        for belt, assessment in assessments.items():
            all_strong_concepts.extend(assessment.strong_concepts)
            all_weak_concepts.extend(assessment.weak_concepts)

            if assessment.mastery_level in [MasteryLevel.MASTERED, MasteryLevel.FULLY_MASTERED]:
                mastered_belts.append(belt)
            elif assessment.mastery_level == MasteryLevel.NEEDS_FULL_COURSE:
                struggling_belts.append(belt)

        # Format strengths
        if mastered_belts:
            strengths.append(
                f"Strong performance in: {', '.join(mastered_belts)}")

        # Count concept frequencies
        from collections import Counter
        strong_counts = Counter(all_strong_concepts)
        weak_counts = Counter(all_weak_concepts)

        for concept, count in strong_counts.most_common(3):
            strengths.append(f"Solid understanding of: {concept}")

        # Format weaknesses
        if struggling_belts:
            weaknesses.append(f"Needs work in: {', '.join(struggling_belts)}")

        for concept, count in weak_counts.most_common(3):
            weaknesses.append(f"Review needed for: {concept}")

        return strengths, weaknesses

    def evaluate(
        self,
        questions: List[Dict],
        answers: List[int]
    ) -> PlacementDecision:
        """
        Evaluate placement test with independent belt assessment.

        Args:
            questions: List of question dictionaries from the test
            answers: List of answer indices provided by the student

        Returns:
            PlacementDecision with which belts to study/skip
        """

        if len(questions) != len(answers):
            raise ValueError(
                f"Questions ({len(questions)}) and answers ({len(answers)}) "
                "must have same length"
            )

        # Get unique belts
        belts = set(q.get('belt') for q in questions if q.get('belt'))

        # Assess each belt independently
        assessments = {}
        for belt in belts:
            assessments[belt] = self._assess_belt(questions, answers, belt)

        # Categorize belts
        belts_to_study = []
        belts_to_skip = []
        belts_to_review = []

        for belt, assessment in assessments.items():
            if assessment.mastery_level == MasteryLevel.FULLY_MASTERED:
                belts_to_skip.append(belt)
            elif assessment.mastery_level == MasteryLevel.MASTERED:
                belts_to_skip.append(belt)
            elif assessment.mastery_level == MasteryLevel.NEEDS_REVIEW:
                belts_to_review.append(belt)
                belts_to_study.append(belt)
            elif assessment.mastery_level == MasteryLevel.NEEDS_FULL_COURSE:
                belts_to_study.append(belt)

        # Calculate study priority
        study_priority = self._calculate_study_priority(assessments)

        # Calculate overall readiness
        if assessments:
            total_weighted = sum(
                a.weighted_score for a in assessments.values())
            max_weighted = sum(
                a.max_weighted_score for a in assessments.values())
            overall_readiness = (
                total_weighted / max_weighted * 100) if max_weighted > 0 else 0
        else:
            overall_readiness = 0

        # Collect insights
        overall_strengths, overall_weaknesses = self._collect_overall_insights(
            assessments)

        # Generate study plan
        study_plan = self._generate_study_plan(assessments, study_priority)

        # Build detailed analysis
        detailed_analysis = {
            'assessments': {
                belt: {
                    'percentage': a.percentage,
                    'mastery_level': a.mastery_level.value,
                    'correct': a.correct_answers,
                    'total': a.total_questions,
                    'by_difficulty': a.by_difficulty,
                    'concepts': {
                        name: {
                            'percentage': c.percentage,
                            'mastered': c.mastered,
                            'correct': c.correct,
                            'total': c.total
                        }
                        for name, c in a.by_concept.items()
                    }
                }
                for belt, a in assessments.items()
            },
            'summary': {
                'total_questions': len(questions),
                'total_correct': sum(a.correct_answers for a in assessments.values()),
                'belts_assessed': len(assessments),
                'belts_mastered': len(belts_to_skip),
                'belts_need_work': len(belts_to_study)
            }
        }

        return PlacementDecision(
            belts_to_study=belts_to_study,
            belts_to_skip=belts_to_skip,
            belts_to_review=belts_to_review,
            study_priority=study_priority,
            total_belts_assessed=len(assessments),
            overall_readiness=overall_readiness,
            belt_assessments=assessments,
            overall_strengths=overall_strengths,
            overall_weaknesses=overall_weaknesses,
            study_plan=study_plan,
            detailed_analysis=detailed_analysis
        )


def evaluate_placement_test(
    questions: List[Dict],
    answers: List[int],
    mastery_threshold: float = 0.70,
    use_weighted_scoring: bool = True
) -> PlacementDecision:
    """
    Convenience function to evaluate a placement test.

    Args:
        questions: List of question dictionaries
        answers: List of student's answer indices (same order as questions)
        mastery_threshold: Minimum percentage to master a belt (0.70 = 70%)
        use_weighted_scoring: Whether to weight by difficulty

    Returns:
        PlacementDecision with which belts to study/skip
    """
    evaluator = IndependentBeltEvaluator(
        mastery_threshold=mastery_threshold,
        use_weighted_scoring=use_weighted_scoring
    )
    return evaluator.evaluate(questions, answers)


def get_placement_summary(result: PlacementDecision) -> Dict:
    """
    Get a simple summary dictionary for API responses.

    Returns:
        Dictionary with key placement information
    """
    return {
        'overall_readiness': round(result.overall_readiness, 1),
        'belts_to_study': result.belts_to_study,
        'belts_to_skip': result.belts_to_skip,
        'belts_to_review': result.belts_to_review,
        'study_priority': [
            {
                'rank': p['rank'],
                'belt': p['belt'],
                'score_percentage': round(p['percentage'] * 100, 1),
                'status': p['mastery_level'],
                'weak_concepts': p['weak_concepts']
            }
            for p in result.study_priority
        ],
        'study_plan': result.study_plan,
        'strengths': result.overall_strengths,
        'weaknesses': result.overall_weaknesses
    }


def print_placement_report(result: PlacementDecision) -> None:
    """Print a formatted placement report"""

    print("\n" + "=" * 70)
    print("📊 INDEPENDENT BELT PLACEMENT REPORT")
    print("=" * 70)

    # Overall Readiness
    readiness = result.overall_readiness
    if readiness >= 80:
        emoji = "🌟"
    elif readiness >= 60:
        emoji = "👍"
    elif readiness >= 40:
        emoji = "📚"
    else:
        emoji = "🎯"

    print(f"\n{emoji} OVERALL READINESS: {readiness:.1f}%")
    print(f"   Belts Assessed: {result.total_belts_assessed}")

    # Placement Decision
    print("\n" + "-" * 70)
    print("🎯 PLACEMENT DECISION")
    print("-" * 70)

    if result.belts_to_skip:
        print(f"\n   ✅ CAN SKIP ({len(result.belts_to_skip)} belts):")
        for belt in result.belts_to_skip:
            assessment = result.belt_assessments[belt]
            print(
                f"      • {belt}: {assessment.percentage:.0%} - {assessment.mastery_level.value}")

    if result.belts_to_review:
        print(f"\n   📖 NEEDS REVIEW ({len(result.belts_to_review)} belts):")
        for belt in result.belts_to_review:
            assessment = result.belt_assessments[belt]
            print(f"      • {belt}: {assessment.percentage:.0%}")
            if assessment.weak_concepts:
                print(
                    f"        Focus on: {', '.join(assessment.weak_concepts)}")

    belts_full_course = [
        b for b in result.belts_to_study
        if b not in result.belts_to_review
    ]
    if belts_full_course:
        print(f"\n   📚 NEEDS FULL COURSE ({len(belts_full_course)} belts):")
        for belt in belts_full_course:
            assessment = result.belt_assessments[belt]
            print(f"      • {belt}: {assessment.percentage:.0%}")

    # Study Priority
    if result.study_priority:
        print("\n" + "-" * 70)
        print("📋 RECOMMENDED STUDY ORDER")
        print("-" * 70)

        for item in result.study_priority:
            print(f"\n   {item['rank']}. {item['belt']}")
            print(f"      Score: {item['percentage']:.0%}")
            print(
                f"      Status: {item['mastery_level'].replace('_', ' ').title()}")
            if item['reasons']:
                print(f"      Why: {'; '.join(item['reasons'])}")
            if item['weak_concepts']:
                print(
                    f"      Weak Concepts: {', '.join(item['weak_concepts'])}")

    # Belt Details
    print("\n" + "-" * 70)
    print("📈 DETAILED BELT ANALYSIS")
    print("-" * 70)

    for belt, assessment in result.belt_assessments.items():
        status_emoji = "✅" if assessment.can_skip else "📚"
        print(f"\n   {status_emoji} {belt}")
        print(f"      Score: {assessment.correct_answers}/{assessment.total_questions} "
              f"({assessment.percentage:.0%})")
        print(
            f"      Level: {assessment.mastery_level.value.replace('_', ' ').title()}")

        # Difficulty breakdown
        if assessment.by_difficulty:
            diff_parts = []
            for diff in sorted(assessment.by_difficulty.keys()):
                stats = assessment.by_difficulty[diff]
                label = IndependentBeltEvaluator.DIFFICULTY_LABELS.get(
                    diff, f'L{diff}')
                pct = (stats['correct'] / stats['total']
                       * 100) if stats['total'] > 0 else 0
                diff_parts.append(
                    f"{label}: {stats['correct']}/{stats['total']} ({pct:.0f}%)")
            print(f"      By Difficulty: {' | '.join(diff_parts)}")

        # Concepts
        if assessment.strong_concepts:
            print(f"      💪 Strong: {', '.join(assessment.strong_concepts)}")
        if assessment.weak_concepts:
            print(f"      ⚠️  Weak: {', '.join(assessment.weak_concepts)}")

    # Study Plan
    if result.study_plan:
        print("\n" + "-" * 70)
        print("📝 PERSONALIZED STUDY PLAN")
        print("-" * 70)
        for item in result.study_plan:
            print(f"\n   {item}")

    # Strengths & Weaknesses
    if result.overall_strengths:
        print("\n   💪 STRENGTHS:")
        for s in result.overall_strengths:
            print(f"      • {s}")

    if result.overall_weaknesses:
        print("\n   ⚠️  AREAS FOR IMPROVEMENT:")
        for w in result.overall_weaknesses:
            print(f"      • {w}")

    print("\n" + "=" * 70)


# =============================================================================
# USAGE EXAMPLE
# =============================================================================

if __name__ == "__main__":
    # Import the generator
    from placement_test_generator import generate_placement_test
    import random

    # Generate test questions
    questions = generate_placement_test(
        questions_dir=".",
        n_questions_per_belt=10,
        easy_pct=0.30,
        medium_pct=0.40,
        hard_pct=0.30,
        belts=["White Belt", "Yellow Belt", "Orange Belt"],
        age_group="6-9",
        language="en",
        # seed=42
    )

    print(f"Generated {len(questions)} questions")

    # Simulate student answers with varying performance per belt
    # random.seed(456)

    belt_skill = {
        "White Belt": 0.55,
        "Yellow Belt": 0.50,
        "Orange Belt": 0.60
    }

    student_answers = []
    for q in questions:
        belt = q.get('belt', '')
        correct_idx = q.get('ans_idx')
        skill = belt_skill.get(belt, 0.5)

        # Adjust skill by difficulty
        difficulty = q.get('difficulty_level', 1)
        adjusted_skill = skill - (difficulty - 1) * 0.15

        if random.random() < adjusted_skill:
            student_answers.append(correct_idx)
        else:
            choices = q.get('choices', [])
            wrong = [i for i in range(len(choices)) if i != correct_idx]
            student_answers.append(random.choice(wrong) if wrong else 0)

    # Evaluate
    result = evaluate_placement_test(
        questions=questions,
        answers=student_answers,
        mastery_threshold=0.70,
        use_weighted_scoring=True
    )

    # Print detailed report
    # print_placement_report(result)
    from rich import print as rp
    # Get simple summary (for API)
    summary = get_placement_summary(result)
    print("\n📦 API Summary:")
    rp(summary)

    # # Direct access examples
    # print("\n📌 Direct Access Examples:")
    # print(f"   result.belts_to_study = {result.belts_to_study}")
    # print(f"   result.belts_to_skip = {result.belts_to_skip}")
    # print(f"   result.overall_readiness = {result.overall_readiness:.1f}%")

Generated 30 questions

📦 API Summary:


{
    'overall_readiness': 28.3,
    'belts_to_study': ['Yellow', 'Orange', 'White'],
    'belts_to_skip': [],
    'belts_to_review': [],
    'study_priority': [
        {
            'rank': 1,
            'belt': 'Yellow',
            'score_percentage': 30.0,
            'status': 'needs_full_course',
            'weak_concepts': [
                'Algorithm',
                'Step Order',
                'Code Awareness',
                'Programming Blocks',
                'Run & Reset',
                'Testing Programs',
                'Story in Programming',
                'Story Structure',
                'Events',
                'Cause and Effect',
                'Program Flow',
                'Choices',
                'Conditionals'
            ]
        },
        {
            'rank': 2,
            'belt': 'White',
            'score_percentage': 30.0,
            'status': 'needs_full_course',
            'weak_concepts': [
                'Binary',
                'Numbers',
                'Screenshot',
                'Sharing',
                'Avoiding Ads',
                'Safety',
                'Password',
                'Strong vs Weak',
                'Secret',
                'Snipping Tool'
            ]
        },
        {
            'rank': 3,
            'belt': 'Orange',
            'score_percentage': 25.0,
            'status': 'needs_full_course',
            'weak_concepts': [
                'Bar-Graph',
                'Reading Data',
                'Text to Speech',
                'Speech to Text',
                'Examples',
                'Count',
                'AI Examples',
                'Variables',
                'Changing Values'
            ]
        }
    ],
    'study_plan': [
        '📚 Start with complete courses for: Yellow, White, Orange',
        '🎯 Focus especially on these concepts: Conditionals, Avoiding Ads, Screenshot, Programming Blocks, 
Bar-Graph',
        '⭐ Recommended to start with: Yellow (Score: 30%)'
    ],
    'strengths': [
        'Solid understanding of: Missing Steps',
        'Solid understanding of: Animation',
        'Solid understanding of: Character Actions'
    ],
    'weaknesses': [
        'Needs work in: Yellow, Orange, White',
        'Review needed for: Algorithm',
        'Review needed for: Step Order',
        'Review needed for: Code Awareness'
    ]
}

In [2]:
"""
Placement Test Evaluation System
================================

Evaluates student placement tests across independent belts (topics/skill areas).
Handles normal evaluation flows and all edge cases with comprehensive tiebreaker logic.

Usage:
    from placement_evaluator import evaluate_placement_test
    
    result = evaluate_placement_test(questions, student_answers)
    summary = result  # Returns API-friendly dictionary
"""

import json
from typing import List, Dict, Optional, Tuple, Any
from dataclasses import dataclass, field
from collections import defaultdict
from enum import Enum
import math


# =============================================================================
# ENUMS
# =============================================================================

class MasteryLevel(Enum):
    """Mastery level for a belt"""
    NOT_ASSESSED = "not_assessed"
    NEEDS_FULL_COURSE = "needs_full_course"
    NEEDS_REVIEW = "needs_review"
    MASTERED = "mastered"
    FULLY_MASTERED = "fully_mastered"


class ConfidenceLevel(Enum):
    """Confidence level based on question count"""
    NONE = "none"
    VERY_LOW = "very_low"
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"


# =============================================================================
# CONFIGURATION
# =============================================================================

DEFAULT_CONFIG = {
    # Mastery thresholds
    "mastery_threshold": 0.70,
    "fully_mastered_threshold": 0.90,
    "review_threshold": 0.40,

    # Scoring
    "use_weighted_scoring": True,
    "difficulty_weights": {1: 1, 2: 2, 3: 3},

    # Concepts
    "concept_mastery_threshold": 0.70,
    "concept_weak_threshold": 0.50,
    "concept_strong_threshold": 0.80,

    # Confidence
    "min_questions_high_confidence": 10,
    "min_questions_medium_confidence": 5,
    "min_questions_low_confidence": 3,

    # Priority calculation weights
    "priority_weights": {
        "base": 100,
        "easy_performance": 30,
        "weak_concepts": 5,
        "hard_performance": 10,
        "belt_importance": 2,
        "full_course_penalty": 15,
        "variance_penalty": 10
    },

    # Default belt importance (lower = more foundational)
    "belt_importance": {
        "White Belt": 1,
        "Yellow Belt": 2,
        "Orange Belt": 3,
        "Green Belt": 4,
        "Blue Belt": 5,
        "Purple Belt": 6,
        "Brown Belt": 7,
        "Black Belt": 8
    }
}


# =============================================================================
# DATA CLASSES
# =============================================================================

@dataclass
class ConceptScore:
    """Score for a specific concept"""
    concept: str
    correct: int
    total: int
    percentage: float
    status: str  # "strong", "adequate", "moderate", "weak"
    mastered: bool


@dataclass
class DifficultyScore:
    """Score for a difficulty level"""
    level: int
    label: str
    correct: int
    total: int
    percentage: float


@dataclass
class BeltAssessment:
    """Complete assessment for a single belt"""
    belt: str
    total_questions: int
    correct_answers: int
    percentage: float
    weighted_score: float
    max_weighted_score: float
    mastery_level: MasteryLevel
    confidence: ConfidenceLevel

    # Breakdown
    by_difficulty: Dict[int, DifficultyScore] = field(default_factory=dict)
    by_concept: Dict[str, ConceptScore] = field(default_factory=dict)

    # Analysis
    strong_concepts: List[str] = field(default_factory=list)
    weak_concepts: List[str] = field(default_factory=list)

    # Anomaly detection
    difficulty_variance: float = 0.0
    has_inverted_difficulty: bool = False

    @property
    def needs_study(self) -> bool:
        return self.mastery_level in [
            MasteryLevel.NEEDS_FULL_COURSE,
            MasteryLevel.NEEDS_REVIEW
        ]

    @property
    def can_skip(self) -> bool:
        return self.mastery_level in [
            MasteryLevel.MASTERED,
            MasteryLevel.FULLY_MASTERED
        ]


@dataclass
class PriorityItem:
    """Priority information for a belt"""
    rank: int
    belt: str
    priority_score: float
    percentage: float
    mastery_level: str
    reasons: List[str]
    weak_concepts: List[str]

    # Tiebreaker details
    easy_percentage: float = 0.0
    hard_percentage: float = 0.0
    weak_concept_count: int = 0
    belt_importance: int = 0


# =============================================================================
# MAIN EVALUATOR CLASS
# =============================================================================

class PlacementTestEvaluator:
    """
    Evaluates placement tests with independent belt assessment.

    Each belt is evaluated independently - no sequential dependency.
    Handles all edge cases and applies tiebreaker logic for equal scores.
    """

    DIFFICULTY_LABELS = {1: "Easy", 2: "Medium", 3: "Hard"}

    def __init__(self, config: Optional[Dict] = None):
        """
        Initialize evaluator with configuration.

        Args:
            config: Optional configuration dictionary (uses defaults if not provided)
        """
        self.config = {**DEFAULT_CONFIG, **(config or {})}
        self._extract_config()

    def _extract_config(self):
        """Extract configuration values for easy access"""
        self.mastery_threshold = self.config["mastery_threshold"]
        self.fully_mastered_threshold = self.config["fully_mastered_threshold"]
        self.review_threshold = self.config["review_threshold"]
        self.use_weighted_scoring = self.config["use_weighted_scoring"]
        self.difficulty_weights = self.config["difficulty_weights"]
        self.concept_mastery_threshold = self.config["concept_mastery_threshold"]
        self.concept_weak_threshold = self.config["concept_weak_threshold"]
        self.concept_strong_threshold = self.config["concept_strong_threshold"]
        self.priority_weights = self.config["priority_weights"]
        self.belt_importance = self.config["belt_importance"]

    # -------------------------------------------------------------------------
    # MASTERY LEVEL DETERMINATION
    # -------------------------------------------------------------------------

    def _determine_mastery_level(self, percentage: float) -> MasteryLevel:
        """Determine mastery level based on percentage score"""
        if percentage >= self.fully_mastered_threshold:
            return MasteryLevel.FULLY_MASTERED
        elif percentage >= self.mastery_threshold:
            return MasteryLevel.MASTERED
        elif percentage >= self.review_threshold:
            return MasteryLevel.NEEDS_REVIEW
        else:
            return MasteryLevel.NEEDS_FULL_COURSE

    def _determine_confidence(self, question_count: int) -> ConfidenceLevel:
        """Determine confidence level based on question count"""
        if question_count >= self.config["min_questions_high_confidence"]:
            return ConfidenceLevel.HIGH
        elif question_count >= self.config["min_questions_medium_confidence"]:
            return ConfidenceLevel.MEDIUM
        elif question_count >= self.config["min_questions_low_confidence"]:
            return ConfidenceLevel.LOW
        elif question_count > 0:
            return ConfidenceLevel.VERY_LOW
        else:
            return ConfidenceLevel.NONE

    def _determine_concept_status(self, percentage: float) -> str:
        """Determine concept status based on percentage"""
        if percentage >= self.concept_strong_threshold:
            return "strong"
        elif percentage >= self.concept_mastery_threshold:
            return "adequate"
        elif percentage >= self.concept_weak_threshold:
            return "moderate"
        else:
            return "weak"

    # -------------------------------------------------------------------------
    # STATISTICAL HELPERS
    # -------------------------------------------------------------------------

    def _calculate_variance(self, values: List[float]) -> float:
        """Calculate variance of a list of values"""
        if len(values) < 2:
            return 0.0
        mean = sum(values) / len(values)
        return sum((x - mean) ** 2 for x in values) / len(values)

    def _calculate_standard_deviation(self, values: List[float]) -> float:
        """Calculate standard deviation"""
        return math.sqrt(self._calculate_variance(values))

    # -------------------------------------------------------------------------
    # BELT ASSESSMENT
    # -------------------------------------------------------------------------

    def _assess_belt(
        self,
        questions: List[Dict],
        answers: List[int],
        belt: str
    ) -> BeltAssessment:
        """
        Assess a single belt independently.

        Calculates scores, analyzes by difficulty and concept,
        and detects anomalies.
        """
        # Filter questions for this belt
        belt_data = [
            (q, a) for q, a in zip(questions, answers)
            if q.get("belt") == belt
        ]

        # Handle no questions case
        if not belt_data:
            return BeltAssessment(
                belt=belt,
                total_questions=0,
                correct_answers=0,
                percentage=0.0,
                weighted_score=0.0,
                max_weighted_score=0.0,
                mastery_level=MasteryLevel.NOT_ASSESSED,
                confidence=ConfidenceLevel.NONE
            )

        # Initialize counters
        total = len(belt_data)
        correct = 0
        weighted_score = 0.0
        max_weighted_score = 0.0

        # Difficulty tracking
        difficulty_stats = defaultdict(lambda: {"correct": 0, "total": 0})

        # Concept tracking
        concept_stats = defaultdict(lambda: {"correct": 0, "total": 0})

        # Process each question
        for question, answer in belt_data:
            difficulty = question.get("difficulty_level", 1)
            concepts = question.get("concepts", [])
            correct_idx = question.get("ans_idx")
            is_correct = (answer == correct_idx)

            # Get weight for difficulty
            weight = self.difficulty_weights.get(difficulty, 1)
            max_weighted_score += weight

            if is_correct:
                correct += 1
                weighted_score += weight

            # Track by difficulty
            difficulty_stats[difficulty]["total"] += 1
            if is_correct:
                difficulty_stats[difficulty]["correct"] += 1

            # Track by concept
            for concept in concepts:
                concept_stats[concept]["total"] += 1
                if is_correct:
                    concept_stats[concept]["correct"] += 1

        # Calculate overall percentage
        if self.use_weighted_scoring and max_weighted_score > 0:
            percentage = weighted_score / max_weighted_score
        else:
            percentage = correct / total if total > 0 else 0.0

        # Process difficulty scores
        by_difficulty = {}
        difficulty_percentages = []

        for diff_level in sorted(difficulty_stats.keys()):
            stats = difficulty_stats[diff_level]
            diff_pct = stats["correct"] / \
                stats["total"] if stats["total"] > 0 else 0.0
            difficulty_percentages.append(diff_pct)

            by_difficulty[diff_level] = DifficultyScore(
                level=diff_level,
                label=self.DIFFICULTY_LABELS.get(
                    diff_level, f"Level {diff_level}"),
                correct=stats["correct"],
                total=stats["total"],
                percentage=diff_pct
            )

        # Process concept scores
        by_concept = {}
        strong_concepts = []
        weak_concepts = []

        for concept, stats in concept_stats.items():
            if stats["total"] > 0:
                concept_pct = stats["correct"] / stats["total"]
                status = self._determine_concept_status(concept_pct)
                mastered = concept_pct >= self.concept_mastery_threshold

                by_concept[concept] = ConceptScore(
                    concept=concept,
                    correct=stats["correct"],
                    total=stats["total"],
                    percentage=concept_pct,
                    status=status,
                    mastered=mastered
                )

                if status == "strong":
                    strong_concepts.append(concept)
                elif status == "weak":
                    weak_concepts.append(concept)

        # Calculate difficulty variance
        difficulty_variance = self._calculate_variance(difficulty_percentages)

        # Detect inverted difficulty pattern
        has_inverted_difficulty = False
        easy_stats = difficulty_stats.get(1, {"correct": 0, "total": 0})
        hard_stats = difficulty_stats.get(3, {"correct": 0, "total": 0})

        if easy_stats["total"] > 0 and hard_stats["total"] > 0:
            easy_pct = easy_stats["correct"] / easy_stats["total"]
            hard_pct = hard_stats["correct"] / hard_stats["total"]
            # Inverted if hard is significantly better than easy
            if hard_pct > easy_pct + 0.25:
                has_inverted_difficulty = True

        # Determine mastery level and confidence
        mastery_level = self._determine_mastery_level(percentage)
        confidence = self._determine_confidence(total)

        return BeltAssessment(
            belt=belt,
            total_questions=total,
            correct_answers=correct,
            percentage=percentage,
            weighted_score=weighted_score,
            max_weighted_score=max_weighted_score,
            mastery_level=mastery_level,
            confidence=confidence,
            by_difficulty=by_difficulty,
            by_concept=by_concept,
            strong_concepts=strong_concepts,
            weak_concepts=weak_concepts,
            difficulty_variance=difficulty_variance,
            has_inverted_difficulty=has_inverted_difficulty
        )

    # -------------------------------------------------------------------------
    # PRIORITY CALCULATION
    # -------------------------------------------------------------------------

    def _get_belt_importance(self, belt: str) -> int:
        """Get importance value for a belt (lower = more important)"""
        return self.belt_importance.get(belt, 99)

    def _get_max_belt_importance(self, belts: List[str]) -> int:
        """Get maximum belt importance from list"""
        if not belts:
            return 10
        return max(self._get_belt_importance(b) for b in belts)

    def _calculate_priority_score(
        self,
        assessment: BeltAssessment,
        max_importance: int
    ) -> Tuple[float, List[str]]:
        """
        Calculate priority score for a belt.

        Higher score = Higher study priority

        Returns:
            Tuple of (priority_score, list of reasons)
        """
        reasons = []
        weights = self.priority_weights

        # Base score: inverse of percentage
        base_score = (1 - assessment.percentage) * weights["base"]

        # Adjustment 1: Easy question performance
        easy_diff = assessment.by_difficulty.get(1)
        if easy_diff and easy_diff.total > 0:
            easy_pct = easy_diff.percentage
            adj_easy = (1 - easy_pct) * weights["easy_performance"]
            if easy_pct < 0.70:
                reasons.append("Fundamental concepts need attention")
        else:
            adj_easy = 0
            easy_pct = 0

        # Adjustment 2: Weak concept count
        weak_count = len(assessment.weak_concepts)
        adj_concepts = weak_count * weights["weak_concepts"]
        if weak_count > 0:
            reasons.append(f"{weak_count} concept(s) need focused study")

        # Adjustment 3: Hard question performance
        hard_diff = assessment.by_difficulty.get(3)
        if hard_diff and hard_diff.total > 0:
            hard_pct = hard_diff.percentage
            adj_hard = (1 - hard_pct) * weights["hard_performance"]
        else:
            adj_hard = 0
            hard_pct = 0

        # Adjustment 4: Belt importance
        belt_importance = self._get_belt_importance(assessment.belt)
        adj_importance = (max_importance - belt_importance) * \
            weights["belt_importance"]
        if belt_importance <= 2:
            reasons.append("Foundational belt - prioritize for strong base")

        # Adjustment 5: Mastery level penalty
        if assessment.mastery_level == MasteryLevel.NEEDS_FULL_COURSE:
            adj_level = weights["full_course_penalty"]
            reasons.append("Requires comprehensive course study")
        else:
            adj_level = 0
            reasons.append("Review and reinforcement recommended")

        # Adjustment 6: High variance penalty
        if assessment.difficulty_variance > 0.1:
            adj_variance = weights["variance_penalty"]
            reasons.append("Inconsistent performance across difficulty levels")
        else:
            adj_variance = 0

        # Calculate total
        priority_score = (
            base_score +
            adj_easy +
            adj_concepts +
            adj_hard +
            adj_importance +
            adj_level +
            adj_variance
        )

        return priority_score, reasons

    def _calculate_study_priority(
        self,
        assessments: Dict[str, BeltAssessment]
    ) -> List[PriorityItem]:
        """
        Calculate priority order for all belts needing study.

        Applies tiebreaker logic when scores are equal.
        """
        # Filter to only belts needing study
        study_belts = {
            belt: assess for belt, assess in assessments.items()
            if assess.needs_study
        }

        if not study_belts:
            return []

        # Get max importance for calculations
        max_importance = self._get_max_belt_importance(
            list(study_belts.keys()))

        # Calculate priority for each belt
        priority_items = []

        for belt, assessment in study_belts.items():
            priority_score, reasons = self._calculate_priority_score(
                assessment, max_importance
            )

            # Get difficulty percentages for tiebreaking
            easy_diff = assessment.by_difficulty.get(1)
            hard_diff = assessment.by_difficulty.get(3)

            easy_pct = easy_diff.percentage if easy_diff else 0.0
            hard_pct = hard_diff.percentage if hard_diff else 0.0

            priority_items.append(PriorityItem(
                rank=0,  # Will be set after sorting
                belt=belt,
                priority_score=priority_score,
                percentage=assessment.percentage,
                mastery_level=assessment.mastery_level.value,
                reasons=reasons,
                weak_concepts=assessment.weak_concepts.copy(),
                easy_percentage=easy_pct,
                hard_percentage=hard_pct,
                weak_concept_count=len(assessment.weak_concepts),
                belt_importance=self._get_belt_importance(belt)
            ))

        # Sort with tiebreakers
        priority_items = self._sort_with_tiebreakers(priority_items)

        # Assign ranks
        for i, item in enumerate(priority_items, 1):
            item.rank = i

        return priority_items

    def _sort_with_tiebreakers(
        self,
        items: List[PriorityItem]
    ) -> List[PriorityItem]:
        """
        Sort priority items with comprehensive tiebreaker logic.

        Tiebreaker order:
        1. Priority score (higher = first)
        2. Easy question percentage (lower = first)
        3. Weak concept count (higher = first)
        4. Hard question percentage (lower = first)
        5. Belt importance (lower number = first)
        6. Alphabetical by belt name
        """
        return sorted(
            items,
            key=lambda x: (
                -x.priority_score,           # Higher score first
                x.easy_percentage,           # Lower easy % first
                -x.weak_concept_count,       # More weak concepts first
                x.hard_percentage,           # Lower hard % first
                x.belt_importance,           # Lower importance number first
                x.belt                       # Alphabetical
            )
        )

    # -------------------------------------------------------------------------
    # FLAG DETECTION
    # -------------------------------------------------------------------------

    def _detect_flags(
        self,
        assessments: Dict[str, BeltAssessment]
    ) -> List[str]:
        """Detect anomalies and edge cases"""
        flags = []

        if not assessments:
            flags.append("NO_ASSESSMENTS")
            return flags

        valid_assessments = [
            a for a in assessments.values()
            if a.mastery_level != MasteryLevel.NOT_ASSESSED
        ]

        if not valid_assessments:
            flags.append("NO_VALID_ASSESSMENTS")
            return flags

        percentages = [a.percentage for a in valid_assessments]

        # Check for all zero scores
        if all(p == 0 for p in percentages):
            flags.append("ALL_ZERO_SCORE")

        # Check for all perfect scores
        if all(p == 1.0 for p in percentages):
            flags.append("ALL_PERFECT_SCORE")

        # Check for all equal scores
        if len(set(percentages)) == 1 and len(percentages) > 1:
            flags.append("ALL_EQUAL_SCORES")

        # Check for low confidence
        low_confidence_count = sum(
            1 for a in valid_assessments
            if a.confidence in [ConfidenceLevel.LOW, ConfidenceLevel.VERY_LOW]
        )
        if low_confidence_count > 0:
            flags.append("LOW_CONFIDENCE_ASSESSMENTS")

        # Check for inverted difficulty patterns
        inverted_count = sum(
            1 for a in valid_assessments
            if a.has_inverted_difficulty
        )
        if inverted_count > 0:
            flags.append("INVERTED_DIFFICULTY_PATTERN")

        # Check for high variance
        high_variance_count = sum(
            1 for a in valid_assessments
            if a.difficulty_variance > 0.15
        )
        if high_variance_count > 0:
            flags.append("HIGH_VARIANCE_DETECTED")

        # Check for single belt
        if len(valid_assessments) == 1:
            flags.append("SINGLE_BELT_ASSESSED")

        # Check for borderline scores
        thresholds = [
            self.mastery_threshold,
            self.fully_mastered_threshold,
            self.review_threshold
        ]
        for a in valid_assessments:
            for threshold in thresholds:
                if abs(a.percentage - threshold) < 0.01:
                    flags.append("BORDERLINE_SCORE")
                    break

        return list(set(flags))  # Remove duplicates

    # -------------------------------------------------------------------------
    # INSIGHTS GENERATION
    # -------------------------------------------------------------------------

    def _generate_strengths(
        self,
        assessments: Dict[str, BeltAssessment]
    ) -> List[str]:
        """Generate list of strengths"""
        strengths = []

        # Mastered belts
        mastered_belts = [
            belt for belt, a in assessments.items()
            if a.can_skip
        ]
        if mastered_belts:
            strengths.append(
                f"Strong performance in: {', '.join(mastered_belts)}")

        # Strong concepts across all belts
        all_strong = []
        for a in assessments.values():
            all_strong.extend(a.strong_concepts)

        # Count frequencies
        concept_counts = defaultdict(int)
        for concept in all_strong:
            concept_counts[concept] += 1

        # Top strong concepts
        top_strong = sorted(
            concept_counts.items(),
            key=lambda x: x[1],
            reverse=True
        )[:3]

        for concept, count in top_strong:
            strengths.append(f"Solid understanding of: {concept}")

        # Check difficulty strengths
        diff_totals = defaultdict(lambda: {"correct": 0, "total": 0})
        for a in assessments.values():
            for diff, score in a.by_difficulty.items():
                diff_totals[diff]["correct"] += score.correct
                diff_totals[diff]["total"] += score.total

        for diff, stats in diff_totals.items():
            if stats["total"] > 0:
                pct = stats["correct"] / stats["total"]
                if pct >= 0.85:
                    label = self.DIFFICULTY_LABELS.get(diff, f"Level {diff}")
                    strengths.append(
                        f"Excellent on {label} questions ({pct:.0%})")

        return strengths

    def _generate_weaknesses(
        self,
        assessments: Dict[str, BeltAssessment]
    ) -> List[str]:
        """Generate list of weaknesses"""
        weaknesses = []

        # Belts needing full course
        full_course_belts = [
            belt for belt, a in assessments.items()
            if a.mastery_level == MasteryLevel.NEEDS_FULL_COURSE
        ]
        if full_course_belts:
            weaknesses.append(
                f"Needs comprehensive work in: {', '.join(full_course_belts)}")

        # Weak concepts across all belts
        all_weak = []
        for a in assessments.values():
            all_weak.extend(a.weak_concepts)

        # Count frequencies
        concept_counts = defaultdict(int)
        for concept in all_weak:
            concept_counts[concept] += 1

        # Top weak concepts
        top_weak = sorted(
            concept_counts.items(),
            key=lambda x: x[1],
            reverse=True
        )[:3]

        for concept, count in top_weak:
            weaknesses.append(f"Review needed for: {concept}")

        # Check difficulty weaknesses
        diff_totals = defaultdict(lambda: {"correct": 0, "total": 0})
        for a in assessments.values():
            for diff, score in a.by_difficulty.items():
                diff_totals[diff]["correct"] += score.correct
                diff_totals[diff]["total"] += score.total

        for diff, stats in diff_totals.items():
            if stats["total"] > 0:
                pct = stats["correct"] / stats["total"]
                if pct < 0.40:
                    label = self.DIFFICULTY_LABELS.get(diff, f"Level {diff}")
                    weaknesses.append(
                        f"Struggles with {label} questions ({pct:.0%})")

        return weaknesses

    def _generate_study_plan(
        self,
        study_priority: List[PriorityItem],
        belts_to_review: List[str],
        flags: List[str]
    ) -> List[str]:
        """Generate personalized study plan"""
        plan = []

        # Handle edge cases first
        if "ALL_PERFECT_SCORE" in flags:
            plan.append(
                "🌟 Perfect performance! You've fully mastered all assessed areas.")
            plan.append(
                "Consider advancing to higher-level content or mentoring others.")
            return plan

        if "ALL_ZERO_SCORE" in flags:
            plan.append("📚 Comprehensive study recommended for all belts.")
            plan.append(
                "⚠️ Note: Consider verifying test was completed correctly.")

        if not study_priority:
            plan.append(
                "🎉 Excellent! You've demonstrated mastery across all assessed belts.")
            plan.append(
                "Consider exploring advanced topics or helping other learners.")
            return plan

        # Separate full course and review belts
        full_course = [
            p.belt for p in study_priority
            if p.mastery_level == "needs_full_course"
        ]
        review_only = [
            p.belt for p in study_priority
            if p.mastery_level == "needs_review"
        ]

        if full_course:
            plan.append(
                f"📚 Start with complete courses for: {', '.join(full_course)}")

        if review_only:
            plan.append(
                f"📖 Review materials recommended for: {', '.join(review_only)}")

        # Collect all weak concepts
        all_weak_concepts = []
        for p in study_priority:
            all_weak_concepts.extend(p.weak_concepts)

        unique_weak = list(dict.fromkeys(all_weak_concepts))[:5]
        if unique_weak:
            plan.append(f"🎯 Focus especially on: {', '.join(unique_weak)}")

        # Top priority recommendation
        if study_priority:
            top = study_priority[0]
            plan.append(
                f"⭐ Recommended to start with: {top.belt} "
                f"(Score: {top.percentage:.0%})"
            )

        # Add flag-based recommendations
        if "INVERTED_DIFFICULTY_PATTERN" in flags:
            plan.append(
                "💡 Unusual pattern detected: Review fundamental concepts carefully.")

        if "LOW_CONFIDENCE_ASSESSMENTS" in flags:
            plan.append(
                "ℹ️ Some assessments based on few questions - consider additional testing.")

        return plan

    # -------------------------------------------------------------------------
    # MAIN EVALUATION METHOD
    # -------------------------------------------------------------------------

    def evaluate(
        self,
        questions: List[Dict],
        answers: List[int]
    ) -> Dict[str, Any]:
        """
        Evaluate placement test and return API-friendly summary.

        Args:
            questions: List of question dictionaries from the test
            answers: List of answer indices provided by the student

        Returns:
            Dictionary with placement decision and detailed analysis
        """
        # Validate inputs
        if len(questions) != len(answers):
            raise ValueError(
                f"Questions ({len(questions)}) and answers ({len(answers)}) "
                "must have same length"
            )

        if not questions:
            return self._empty_result()

        # Get unique belts
        belts = set(q.get("belt") for q in questions if q.get("belt"))

        if not belts:
            return self._empty_result()

        # Assess each belt independently
        assessments: Dict[str, BeltAssessment] = {}
        for belt in belts:
            assessments[belt] = self._assess_belt(questions, answers, belt)

        # Categorize belts
        belts_to_study = []
        belts_to_skip = []
        belts_to_review = []

        for belt, assessment in assessments.items():
            if assessment.mastery_level == MasteryLevel.NOT_ASSESSED:
                continue
            elif assessment.can_skip:
                belts_to_skip.append(belt)
            elif assessment.mastery_level == MasteryLevel.NEEDS_REVIEW:
                belts_to_review.append(belt)
                belts_to_study.append(belt)
            elif assessment.mastery_level == MasteryLevel.NEEDS_FULL_COURSE:
                belts_to_study.append(belt)

        # Calculate study priority
        study_priority = self._calculate_study_priority(assessments)

        # Calculate overall readiness
        valid_assessments = [
            a for a in assessments.values()
            if a.mastery_level != MasteryLevel.NOT_ASSESSED
        ]

        if valid_assessments:
            total_weighted = sum(a.weighted_score for a in valid_assessments)
            max_weighted = sum(a.max_weighted_score for a in valid_assessments)
            overall_readiness = (
                total_weighted / max_weighted * 100) if max_weighted > 0 else 0
        else:
            overall_readiness = 0

        # Detect flags
        flags = self._detect_flags(assessments)

        # Generate insights
        strengths = self._generate_strengths(assessments)
        weaknesses = self._generate_weaknesses(assessments)
        study_plan = self._generate_study_plan(
            study_priority, belts_to_review, flags)

        # Build API summary
        return {
            "overall_readiness": round(overall_readiness, 1),
            "total_questions": len(questions),
            "total_correct": sum(a.correct_answers for a in valid_assessments),
            "belts_assessed": len(valid_assessments),

            "belts_to_study": belts_to_study,
            "belts_to_skip": belts_to_skip,
            "belts_to_review": belts_to_review,

            "study_priority": [
                {
                    "rank": p.rank,
                    "belt": p.belt,
                    "score_percentage": round(p.percentage * 100, 1),
                    "status": p.mastery_level,
                    "reasons": p.reasons,
                    "weak_concepts": p.weak_concepts,
                    "priority_score": round(p.priority_score, 2)
                }
                for p in study_priority
            ],

            "belt_details": {
                belt: {
                    "score_percentage": round(a.percentage * 100, 1),
                    "correct": a.correct_answers,
                    "total": a.total_questions,
                    "status": a.mastery_level.value,
                    "confidence": a.confidence.value,
                    "by_difficulty": {
                        str(d): {
                            "label": s.label,
                            "correct": s.correct,
                            "total": s.total,
                            "percentage": round(s.percentage * 100, 1)
                        }
                        for d, s in a.by_difficulty.items()
                    },
                    "strong_concepts": a.strong_concepts,
                    "weak_concepts": a.weak_concepts
                }
                for belt, a in assessments.items()
                if a.mastery_level != MasteryLevel.NOT_ASSESSED
            },

            "study_plan": study_plan,
            "strengths": strengths,
            "weaknesses": weaknesses,
            "flags": flags
        }

    def _empty_result(self) -> Dict[str, Any]:
        """Return empty result for edge cases"""
        return {
            "overall_readiness": 0,
            "total_questions": 0,
            "total_correct": 0,
            "belts_assessed": 0,
            "belts_to_study": [],
            "belts_to_skip": [],
            "belts_to_review": [],
            "study_priority": [],
            "belt_details": {},
            "study_plan": ["No questions available for assessment."],
            "strengths": [],
            "weaknesses": [],
            "flags": ["NO_DATA"]
        }


# =============================================================================
# CONVENIENCE FUNCTION
# =============================================================================

def evaluate_placement_test(
    questions: List[Dict],
    answers: List[int],
    config: Optional[Dict] = None
) -> Dict[str, Any]:
    """
    Evaluate a placement test and return API-friendly summary.

    Args:
        questions: List of question dictionaries from the test
        answers: List of student's answer indices (same order as questions)
        config: Optional configuration dictionary

    Returns:
        Dictionary with placement decision and detailed analysis

    Example:
        >>> result = evaluate_placement_test(questions, student_answers)
        >>> print(result["belts_to_study"])
        ['Yellow Belt', 'Orange Belt']
        >>> print(result["overall_readiness"])
        56.7
    """
    evaluator = PlacementTestEvaluator(config)
    return evaluator.evaluate(questions, answers)


# =============================================================================
# PRETTY PRINT FUNCTION
# =============================================================================

def print_evaluation_report(result: Dict[str, Any]) -> None:
    """Print a formatted evaluation report"""

    print("\n" + "=" * 70)
    print("📊 PLACEMENT TEST EVALUATION REPORT")
    print("=" * 70)

    # Overall Readiness
    readiness = result["overall_readiness"]
    if readiness >= 80:
        emoji = "🌟"
    elif readiness >= 60:
        emoji = "👍"
    elif readiness >= 40:
        emoji = "📚"
    else:
        emoji = "🎯"

    print(f"\n{emoji} OVERALL READINESS: {readiness}%")
    print(
        f"   Questions: {result['total_correct']}/{result['total_questions']} correct")
    print(f"   Belts Assessed: {result['belts_assessed']}")

    # Flags
    if result["flags"]:
        print(f"\n   ⚠️ Flags: {', '.join(result['flags'])}")

    # Placement Decision
    print("\n" + "-" * 70)
    print("🎯 PLACEMENT DECISION")
    print("-" * 70)

    if result["belts_to_skip"]:
        print(f"\n   ✅ CAN SKIP ({len(result['belts_to_skip'])} belt(s)):")
        for belt in result["belts_to_skip"]:
            details = result["belt_details"].get(belt, {})
            pct = details.get("score_percentage", 0)
            status = details.get("status", "unknown")
            print(f"      • {belt}: {pct}% - {status}")

    if result["belts_to_review"]:
        print(
            f"\n   📖 NEEDS REVIEW ({len(result['belts_to_review'])} belt(s)):")
        for belt in result["belts_to_review"]:
            details = result["belt_details"].get(belt, {})
            pct = details.get("score_percentage", 0)
            weak = details.get("weak_concepts", [])
            print(f"      • {belt}: {pct}%")
            if weak:
                print(f"        Focus on: {', '.join(weak)}")

    full_course = [
        b for b in result["belts_to_study"]
        if b not in result["belts_to_review"]
    ]
    if full_course:
        print(f"\n   📚 NEEDS FULL COURSE ({len(full_course)} belt(s)):")
        for belt in full_course:
            details = result["belt_details"].get(belt, {})
            pct = details.get("score_percentage", 0)
            print(f"      • {belt}: {pct}%")

    # Study Priority
    if result["study_priority"]:
        print("\n" + "-" * 70)
        print("📋 RECOMMENDED STUDY ORDER")
        print("-" * 70)

        for item in result["study_priority"]:
            print(f"\n   {item['rank']}. {item['belt']}")
            print(f"      Score: {item['score_percentage']}%")
            print(f"      Status: {item['status'].replace('_', ' ').title()}")
            print(f"      Priority Score: {item['priority_score']}")
            if item["reasons"]:
                print(f"      Why: {'; '.join(item['reasons'])}")
            if item["weak_concepts"]:
                print(f"      Focus Areas: {', '.join(item['weak_concepts'])}")

    # Belt Details
    print("\n" + "-" * 70)
    print("📈 DETAILED BELT ANALYSIS")
    print("-" * 70)

    for belt, details in result["belt_details"].items():
        status_emoji = "✅" if belt in result["belts_to_skip"] else "📚"
        print(f"\n   {status_emoji} {belt}")
        print(f"      Score: {details['correct']}/{details['total']} "
              f"({details['score_percentage']}%)")
        print(f"      Status: {details['status'].replace('_', ' ').title()}")
        print(
            f"      Confidence: {details['confidence'].replace('_', ' ').title()}")

        # Difficulty breakdown
        if details["by_difficulty"]:
            diff_parts = []
            for diff_key in sorted(details["by_difficulty"].keys()):
                d = details["by_difficulty"][diff_key]
                diff_parts.append(
                    f"{d['label']}: {d['correct']}/{d['total']} ({d['percentage']}%)"
                )
            print(f"      By Difficulty: {' | '.join(diff_parts)}")

        if details["strong_concepts"]:
            print(f"      💪 Strong: {', '.join(details['strong_concepts'])}")
        if details["weak_concepts"]:
            print(f"      ⚠️ Weak: {', '.join(details['weak_concepts'])}")

    # Study Plan
    if result["study_plan"]:
        print("\n" + "-" * 70)
        print("📝 PERSONALIZED STUDY PLAN")
        print("-" * 70)
        for item in result["study_plan"]:
            print(f"\n   {item}")

    # Strengths & Weaknesses
    if result["strengths"]:
        print("\n   💪 STRENGTHS:")
        for s in result["strengths"]:
            print(f"      • {s}")

    if result["weaknesses"]:
        print("\n   ⚠️ AREAS FOR IMPROVEMENT:")
        for w in result["weaknesses"]:
            print(f"      • {w}")

    print("\n" + "=" * 70)

In [3]:
from rich import print as rp

In [6]:
# =============================================================================
# USAGE EXAMPLE
# =============================================================================

from placement_test_generator import generate_placement_test

# Generate test questions
questions = generate_placement_test(
    questions_dir=".",
    n_questions_per_belt=10,
    easy_pct=0.30,
    medium_pct=0.40,
    hard_pct=0.30,
    belts=["White Belt", "Yellow Belt", "Orange Belt"],
    age_group="6-9",
    language="en",
    # seed=42
)

print(f"Generated {len(questions)} questions")

# ==========================================================================
# TEST SCENARIO 1: Varied Performance
# ==========================================================================

print("\n" + "=" * 70)
print("TEST SCENARIO 1: VARIED PERFORMANCE")
print("=" * 70)

# Simulate: Good at White, Medium at Yellow, Poor at Orange
belt_skill = {
    "White Belt": 0.85,
    "Yellow Belt": 0.55,
    "Orange Belt": 0.30
}


student_answers_1 = []
for q in questions:
    belt = q.get("belt", "")
    correct_idx = q.get("ans_idx")
    skill = belt_skill.get(belt, 0.5)
    difficulty = q.get("difficulty_level", 1)
    adjusted_skill = skill - (difficulty - 1) * 0.10

    if random.random() < adjusted_skill:
        student_answers_1.append(correct_idx)
    else:
        choices = q.get("choices", [])
        wrong = [i for i in range(len(choices)) if i != correct_idx]
        student_answers_1.append(random.choice(wrong) if wrong else 0)

result_1 = evaluate_placement_test(questions, student_answers_1)

# print_evaluation_report(result_1)
print("\n📦 API SUMMARY (JSON):")

rp(result_1)

Generated 30 questions

TEST SCENARIO 1: VARIED PERFORMANCE

📦 API SUMMARY (JSON):


{
    'overall_readiness': 36.7,
    'total_questions': 30,
    'total_correct': 12,
    'belts_assessed': 3,
    'belts_to_study': ['Yellow', 'Orange', 'White'],
    'belts_to_skip': [],
    'belts_to_review': ['Orange'],
    'study_priority': [
        {
            'rank': 1,
            'belt': 'Yellow',
            'score_percentage': 25.0,
            'status': 'needs_full_course',
            'reasons': [
                'Fundamental concepts need attention',
                '11 concept(s) need focused study',
                'Requires comprehensive course study'
            ],
            'weak_concepts': [
                'Story in Programming',
                'Character Dialogue',
                'Algorithm',
                'Problem Solving',
                'Debugging',
                'Complex Actions',
                'Steps',
                'Events',
                'Cause and Effect',
                'Beginning',
                'Missing Steps'
            ],
            'priority_score': 175.0
        },
        {
            'rank': 2,
            'belt': 'White',
            'score_percentage': 30.0,
            'status': 'needs_full_course',
            'reasons': [
                'Fundamental concepts need attention',
                '12 concept(s) need focused study',
                'Requires comprehensive course study'
            ],
            'weak_concepts': [
                'Create Folder',
                'Steps',
                'Language Switch',
                'Alt+Shift',
                'Ads',
                'Danger',
                'Minimize',
                'Windows',
                'Save',
                'Importance',
                'Snipping Tool',
                'Screenshot'
            ],
            'priority_score': 165.0
        },
        {
            'rank': 3,
            'belt': 'Orange',
            'score_percentage': 55.0,
            'status': 'needs_review',
            'reasons': [
                'Fundamental concepts need attention',
                '6 concept(s) need focused study',
                'Review and reinforcement recommended'
            ],
            'weak_concepts': [
                'Speech to Text',
                'Voice Recognition',
                'AI Examples',
                'Data Analysis',
                'Machine Learning',
                'Brain'
            ],
            'priority_score': 98.33
        }
    ],
    'belt_details': {
        'Yellow': {
            'score_percentage': 25.0,
            'correct': 3,
            'total': 10,
            'status': 'needs_full_course',
            'confidence': 'high',
            'by_difficulty': {
                '1': {'label': 'Easy', 'correct': 1, 'total': 3, 'percentage': 33.3},
                '2': {'label': 'Medium', 'correct': 2, 'total': 4, 'percentage': 50.0},
                '3': {'label': 'Hard', 'correct': 0, 'total': 3, 'percentage': 0.0}
            },
            'strong_concepts': ['Order of Instructions', 'Movement Effects', 'Repeat', 'Loops', 'Efficiency'],
            'weak_concepts': [
                'Story in Programming',
                'Character Dialogue',
                'Algorithm',
                'Problem Solving',
                'Debugging',
                'Complex Actions',
                'Steps',
                'Events',
                'Cause and Effect',
                'Beginning',
                'Missing Steps'
            ]
        },
        'Orange': {
            'score_percentage': 55.0,
            'correct': 5,
            'total': 10,
            'status': 'needs_review',
            'confidence': 'high',
            'by_difficulty': {
                '1': {'label': 'Easy', 'correct': 1, 'total': 3, 'percentage': 33.3},
                '2': {'label': 'Medium', 'correct': 2, 'total': 4, 'percentage': 50.0},
                '3': {'label': 'Hard', 'correct': 2, 'total': 3, 'percentage': 66.7}
            },
            'strong_concepts'

In [8]:

# ==========================================================================
# TEST SCENARIO 2: All Equal Scores (Edge Case)
# ==========================================================================
print("\n" + "=" * 70)
print("TEST SCENARIO 2: ALL EQUAL SCORES (EDGE CASE)")
print("=" * 70)

# random.seed(200)

# Simulate: All belts at ~55%
student_answers_2 = []
for q in questions:
    correct_idx = q.get("ans_idx")
    difficulty = q.get("difficulty_level", 1)

    # Vary by difficulty to test tiebreakers
    belt = q.get("belt", "")
    if belt == "White Belt":
        probs = {1: 0.40, 2: 0.55, 3: 0.70}  # Low easy, high hard
    elif belt == "Yellow Belt":
        probs = {1: 0.70, 2: 0.55, 3: 0.40}  # High easy, low hard
    else:
        probs = {1: 0.55, 2: 0.55, 3: 0.55}  # Balanced

    if random.random() < probs.get(difficulty, 0.55):
        student_answers_2.append(correct_idx)
    else:
        choices = q.get("choices", [])
        wrong = [i for i in range(len(choices)) if i != correct_idx]
        student_answers_2.append(random.choice(wrong) if wrong else 0)

result_2 = evaluate_placement_test(questions, student_answers_2)
rp(result_2)


TEST SCENARIO 2: ALL EQUAL SCORES (EDGE CASE)


{
    'overall_readiness': 60.0,
    'total_questions': 30,
    'total_correct': 18,
    'belts_assessed': 3,
    'belts_to_study': ['Yellow', 'Orange', 'White'],
    'belts_to_skip': [],
    'belts_to_review': ['Yellow', 'Orange', 'White'],
    'study_priority': [
        {
            'rank': 1,
            'belt': 'Yellow',
            'score_percentage': 55.0,
            'status': 'needs_review',
            'reasons': [
                'Fundamental concepts need attention',
                '8 concept(s) need focused study',
                'Review and reinforcement recommended'
            ],
            'weak_concepts': [
                'Animation',
                'Movement Effects',
                'Complex Actions',
                'Steps',
                'Beginning',
                'Repeat',
                'Loops',
                'Efficiency'
            ],
            'priority_score': 108.33
        },
        {
            'rank': 2,
            'belt': 'Orange',
            'score_percentage': 60.0,
            'status': 'needs_review',
            'reasons': [
                'Fundamental concepts need attention',
                '3 concept(s) need focused study',
                'Review and reinforcement recommended'
            ],
            'weak_concepts': ['AI Examples', 'Count', 'Data Science Definition'],
            'priority_score': 68.33
        },
        {
            'rank': 3,
            'belt': 'White',
            'score_percentage': 65.0,
            'status': 'needs_review',
            'reasons': ['6 concept(s) need focused study', 'Review and reinforcement recommended'],
            'weak_concepts': ['Pseudocode', 'Definition', 'Language Switch', 'Alt+Shift', 'Save', 'Importance'],
            'priority_score': 68.33
        }
    ],
    'belt_details': {
        'Yellow': {
            'score_percentage': 55.0,
            'correct': 5,
            'total': 10,
            'status': 'needs_review',
            'confidence': 'high',
            'by_difficulty': {
                '1': {'label': 'Easy', 'correct': 1, 'total': 3, 'percentage': 33.3},
                '2': {'label': 'Medium', 'correct': 2, 'total': 4, 'percentage': 50.0},
                '3': {'label': 'Hard', 'correct': 2, 'total': 3, 'percentage': 66.7}
            },
            'strong_concepts': [
                'Character Dialogue',
                'Order of Instructions',
                'Problem Solving',
                'Debugging',
                'Events',
                'Cause and Effect',
                'Missing Steps'
            ],
            'weak_concepts': [
                'Animation',
                'Movement Effects',
                'Complex Actions',
                'Steps',
                'Beginning',
                'Repeat',
                'Loops',
                'Efficiency'
            ]
        },
        'Orange': {
            'score_percentage': 60.0,
            'correct': 6,
            'total': 10,
            'status': 'needs_review',
            'confidence': 'high',
            'by_difficulty': {
                '1': {'label': 'Easy', 'correct': 2, 'total': 3, 'percentage': 66.7},
                '2': {'label': 'Medium', 'correct': 2, 'total': 4, 'percentage': 50.0},
                '3': {'label': 'Hard', 'correct': 2, 'total': 3, 'percentage': 66.7}
            },
            'strong_concepts': [
                'Speech to Text',
                'Voice Recognition',
                'Storing Data',
                'Data Analysis',
                'Addition',
                'Pic-Graph',
                'Data Visualization',
                'Machine Learning',
                'Brain'
            ],
            'weak_concepts': ['AI Examples', 'Count', 'Data Science Definition']
        },
        'White': {
            'score_percentage': 65.0,
            'correct': 7,
            'total': 10,
            'status': 'needs_review',
            'confidence': 'high',
            '

In [9]:

# ==========================================================================
# TEST SCENARIO 3: All Perfect Scores (Edge Case)
# ==========================================================================
print("\n" + "=" * 70)
print("TEST SCENARIO 3: ALL PERFECT SCORES (EDGE CASE)")
print("=" * 70)

student_answers_3 = [q.get("ans_idx") for q in questions]

result_3 = evaluate_placement_test(questions, student_answers_3)
rp(result_3)


TEST SCENARIO 3: ALL PERFECT SCORES (EDGE CASE)


{
    'overall_readiness': 100.0,
    'total_questions': 30,
    'total_correct': 30,
    'belts_assessed': 3,
    'belts_to_study': [],
    'belts_to_skip': ['Yellow', 'Orange', 'White'],
    'belts_to_review': [],
    'study_priority': [],
    'belt_details': {
        'Yellow': {
            'score_percentage': 100.0,
            'correct': 10,
            'total': 10,
            'status': 'fully_mastered',
            'confidence': 'high',
            'by_difficulty': {
                '1': {'label': 'Easy', 'correct': 3, 'total': 3, 'percentage': 100.0},
                '2': {'label': 'Medium', 'correct': 4, 'total': 4, 'percentage': 100.0},
                '3': {'label': 'Hard', 'correct': 3, 'total': 3, 'percentage': 100.0}
            },
            'strong_concepts': [
                'Story in Programming',
                'Character Dialogue',
                'Sequence',
                'Order of Instructions',
                'Animation',
                'Movement Effects',
                'Algorithm',
                'Problem Solving',
                'Debugging',
                'Complex Actions',
                'Steps',
                'Events',
                'Cause and Effect',
                'Beginning',
                'Missing Steps',
                'Repeat',
                'Loops',
                'Efficiency'
            ],
            'weak_concepts': []
        },
        'Orange': {
            'score_percentage': 100.0,
            'correct': 10,
            'total': 10,
            'status': 'fully_mastered',
            'confidence': 'high',
            'by_difficulty': {
                '1': {'label': 'Easy', 'correct': 3, 'total': 3, 'percentage': 100.0},
                '2': {'label': 'Medium', 'correct': 4, 'total': 4, 'percentage': 100.0},
                '3': {'label': 'Hard', 'correct': 3, 'total': 3, 'percentage': 100.0}
            },
            'strong_concepts': [
                'Variables',
                'Speech to Text',
                'Voice Recognition',
                'AI Examples',
                'Sum',
                'Count',
                'Storing Data',
                'Data Analysis',
                'Addition',
                'Data Science Definition',
                'Pic-Graph',
                'Data Visualization',
                'Machine Learning',
                'Brain'
            ],
            'weak_concepts': []
        },
        'White': {
            'score_percentage': 100.0,
            'correct': 10,
            'total': 10,
            'status': 'fully_mastered',
            'confidence': 'high',
            'by_difficulty': {
                '1': {'label': 'Easy', 'correct': 3, 'total': 3, 'percentage': 100.0},
                '2': {'label': 'Medium', 'correct': 4, 'total': 4, 'percentage': 100.0},
                '3': {'label': 'Hard', 'correct': 3, 'total': 3, 'percentage': 100.0}
            },
            'strong_concepts': [
                'Create Folder',
                'Steps',
                'Pseudocode',
                'Definition',
                'Search',
                'Browser',
                'Internet Safety',
                '3 Rules',
                'Cut',
                'Ctrl+X',
                'Language Switch',
                'Alt+Shift',
                'Ads',
                'Danger',
                'Minimize',
                'Windows',
                'Save',
                'Importance',
                'Snipping Tool',
                'Screenshot'
            ],
            'weak_concepts': []
        }
    },
    'study_plan': [
        "🌟 Perfect performance! You've fully mastered all assessed areas.",
        'Consider advancing to higher-level content or mentoring others.'
    ],
    'strengths': [
        'Strong performance in: Yellow, Orange, White',
        'Solid understanding of: Steps',
        'Solid understanding of: Story in Programming',
        'Solid understanding of: Character Dialogue',

In [10]:

# ==========================================================================
# TEST SCENARIO 4: All Zero Scores (Edge Case)
# ==========================================================================
print("\n" + "=" * 70)
print("TEST SCENARIO 4: ALL ZERO SCORES (EDGE CASE)")
print("=" * 70)

student_answers_4 = []
for q in questions:
    correct_idx = q.get("ans_idx")
    choices = q.get("choices", [])
    wrong = [i for i in range(len(choices)) if i != correct_idx]
    student_answers_4.append(wrong[0] if wrong else 1)

result_4 = evaluate_placement_test(questions, student_answers_4)
rp(result_4)


TEST SCENARIO 4: ALL ZERO SCORES (EDGE CASE)


{
    'overall_readiness': 0.0,
    'total_questions': 30,
    'total_correct': 0,
    'belts_assessed': 3,
    'belts_to_study': ['Yellow', 'Orange', 'White'],
    'belts_to_skip': [],
    'belts_to_review': [],
    'study_priority': [
        {
            'rank': 1,
            'belt': 'White',
            'score_percentage': 0.0,
            'status': 'needs_full_course',
            'reasons': [
                'Fundamental concepts need attention',
                '20 concept(s) need focused study',
                'Requires comprehensive course study'
            ],
            'weak_concepts': [
                'Create Folder',
                'Steps',
                'Pseudocode',
                'Definition',
                'Search',
                'Browser',
                'Internet Safety',
                '3 Rules',
                'Cut',
                'Ctrl+X',
                'Language Switch',
                'Alt+Shift',
                'Ads',
                'Danger',
                'Minimize',
                'Windows',
                'Save',
                'Importance',
                'Snipping Tool',
                'Screenshot'
            ],
            'priority_score': 255.0
        },
        {
            'rank': 2,
            'belt': 'Yellow',
            'score_percentage': 0.0,
            'status': 'needs_full_course',
            'reasons': [
                'Fundamental concepts need attention',
                '18 concept(s) need focused study',
                'Requires comprehensive course study'
            ],
            'weak_concepts': [
                'Story in Programming',
                'Character Dialogue',
                'Sequence',
                'Order of Instructions',
                'Animation',
                'Movement Effects',
                'Algorithm',
                'Problem Solving',
                'Debugging',
                'Complex Actions',
                'Steps',
                'Events',
                'Cause and Effect',
                'Beginning',
                'Missing Steps',
                'Repeat',
                'Loops',
                'Efficiency'
            ],
            'priority_score': 245.0
        },
        {
            'rank': 3,
            'belt': 'Orange',
            'score_percentage': 0.0,
            'status': 'needs_full_course',
            'reasons': [
                'Fundamental concepts need attention',
                '14 concept(s) need focused study',
                'Requires comprehensive course study'
            ],
            'weak_concepts': [
                'Variables',
                'Speech to Text',
                'Voice Recognition',
                'AI Examples',
                'Sum',
                'Count',
                'Storing Data',
                'Data Analysis',
                'Addition',
                'Data Science Definition',
                'Pic-Graph',
                'Data Visualization',
                'Machine Learning',
                'Brain'
            ],
            'priority_score': 225.0
        }
    ],
    'belt_details': {
        'Yellow': {
            'score_percentage': 0.0,
            'correct': 0,
            'total': 10,
            'status': 'needs_full_course',
            'confidence': 'high',
            'by_difficulty': {
                '1': {'label': 'Easy', 'correct': 0, 'total': 3, 'percentage': 0.0},
                '2': {'label': 'Medium', 'correct': 0, 'total': 4, 'percentage': 0.0},
                '3': {'label': 'Hard', 'correct': 0, 'total': 3, 'percentage': 0.0}
            },
            'strong_concepts': [],
            'weak_concepts': [
                'Story in Programming',
                'Character Dialogue',
                'Sequence',
                'Order of Instructions',
                'Animation',
                'Movement Effects',
                'Algorithm',
                'Problem Solving',
                'De

In [13]:

# ==========================================================================
# TEST SCENARIO 5: Mixed Extreme (Edge Case)
# ==========================================================================
print("\n" + "=" * 70)
print("TEST SCENARIO 5: MIXED EXTREME (100%, 50%, 0%)")
print("=" * 70)

student_answers_5 = []
for q in questions:
    belt = q.get("belt", "")
    correct_idx = q.get("ans_idx")
    choices = q.get("choices", [])

    if belt == "White":
        # 100% correct
        student_answers_5.append(correct_idx)
    elif belt == "Yellow":
        # 50% correct
        if random.random() < 0.5:
            student_answers_5.append(correct_idx)
        else:
            wrong = [i for i in range(len(choices)) if i != correct_idx]
            student_answers_5.append(wrong[0] if wrong else 0)
    else:
        # 0% correct
        wrong = [i for i in range(len(choices)) if i != correct_idx]
        student_answers_5.append(wrong[0] if wrong else 1)

result_5 = evaluate_placement_test(questions, student_answers_5)
rp(result_5)

print("\n" + "=" * 70)
print("ALL TESTS COMPLETED")
print("=" * 70)


TEST SCENARIO 5: MIXED EXTREME (100%, 50%, 0%)


{
    'overall_readiness': 50.0,
    'total_questions': 30,
    'total_correct': 16,
    'belts_assessed': 3,
    'belts_to_study': ['Yellow', 'Orange'],
    'belts_to_skip': ['White'],
    'belts_to_review': ['Yellow'],
    'study_priority': [
        {
            'rank': 1,
            'belt': 'Orange',
            'score_percentage': 0.0,
            'status': 'needs_full_course',
            'reasons': [
                'Fundamental concepts need attention',
                '14 concept(s) need focused study',
                'Requires comprehensive course study'
            ],
            'weak_concepts': [
                'Variables',
                'Speech to Text',
                'Voice Recognition',
                'AI Examples',
                'Sum',
                'Count',
                'Storing Data',
                'Data Analysis',
                'Addition',
                'Data Science Definition',
                'Pic-Graph',
                'Data Visualization',
                'Machine Learning',
                'Brain'
            ],
            'priority_score': 225.0
        },
        {
            'rank': 2,
            'belt': 'Yellow',
            'score_percentage': 50.0,
            'status': 'needs_review',
            'reasons': ['7 concept(s) need focused study', 'Review and reinforcement recommended'],
            'weak_concepts': [
                'Character Dialogue',
                'Complex Actions',
                'Events',
                'Cause and Effect',
                'Repeat',
                'Loops',
                'Efficiency'
            ],
            'priority_score': 91.67
        }
    ],
    'belt_details': {
        'Yellow': {
            'score_percentage': 50.0,
            'correct': 6,
            'total': 10,
            'status': 'needs_review',
            'confidence': 'high',
            'by_difficulty': {
                '1': {'label': 'Easy', 'correct': 3, 'total': 3, 'percentage': 100.0},
                '2': {'label': 'Medium', 'correct': 2, 'total': 4, 'percentage': 50.0},
                '3': {'label': 'Hard', 'correct': 1, 'total': 3, 'percentage': 33.3}
            },
            'strong_concepts': [
                'Order of Instructions',
                'Movement Effects',
                'Algorithm',
                'Problem Solving',
                'Steps',
                'Beginning',
                'Missing Steps'
            ],
            'weak_concepts': [
                'Character Dialogue',
                'Complex Actions',
                'Events',
                'Cause and Effect',
                'Repeat',
                'Loops',
                'Efficiency'
            ]
        },
        'Orange': {
            'score_percentage': 0.0,
            'correct': 0,
            'total': 10,
            'status': 'needs_full_course',
            'confidence': 'high',
            'by_difficulty': {
                '1': {'label': 'Easy', 'correct': 0, 'total': 3, 'percentage': 0.0},
                '2': {'label': 'Medium', 'correct': 0, 'total': 4, 'percentage': 0.0},
                '3': {'label': 'Hard', 'correct': 0, 'total': 3, 'percentage': 0.0}
            },
            'strong_concepts': [],
            'weak_concepts': [
                'Variables',
                'Speech to Text',
                'Voice Recognition',
                'AI Examples',
                'Sum',
                'Count',
                'Storing Data',
                'Data Analysis',
                'Addition',
                'Data Science Definition',
                'Pic-Graph',
                'Data Visualization',
                'Machine Learning',
                'Brain'
            ]
        },
        'White': {
            'score_percentage': 100.0,
            'correct': 10,
            'total': 10,
            'status': 'fully_mastered',
            'confidence': 'high',
            'by_difficulty': {
                '1': {'label': 'E


ALL TESTS COMPLETED
